In [ ]:
!pip install -q groq python-dotenv pydantic ollama chromadb langchain-text-splitters tiktoken

In [ ]:
import os
import json
import time
import tiktoken
import ollama
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
load_dotenv()
cliente_groq = Groq()
MODELO_GROQ = "qwen/qwen3.8-27b"
MODELO_OLLAMA = "qwen3:8b"
BACKEND_ACTUAL = "groq"
MODELO_EMBEDDINGS = "qwen3-embedding:8b"
RAG_CHUNK_SIZE = 1000
RAG_OVERLAP_PCT = 80

In [ ]:
cliente_chroma = chromadb.PersistentClient(path="./chroma_db")
funcion_embeddings = OllamaEmbeddingFunction(
    model_name=MODELO_EMBEDDINGS,
    url="http://localhost:11434/api/embeddings"
)

coleccion_politicas = cliente_chroma.get_or_create_collection(
    name="politicas_ecommerce",
    embedding_function=funcion_embeddings
)

INDEXACION_REALIZADA = False

In [ ]:
try:
    prompt_urgencia = open("prompt_urgencia.md", "r", encoding="utf-8").read()
    prompt_categoria = open("prompt_categoria.md", "r", encoding="utf-8").read()
    prompt_accion = open("prompt_accion.md", "r", encoding="utf-8").read()
    prompt_texto = open("prompt_texto.md", "r", encoding="utf-8").read()
    prompt_maestro = open("prompt_maestro.md", "r", encoding="utf-8").read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

try:
    politicas_ecommerce = open("politicas_ecommerce_integral.md", "r", encoding="utf-8").read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer las políticas: {e}")

In [ ]:
class Metadatos(BaseModel):
    categoria: str
    nivel_urgencia: str

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

In [ ]:
def llamar_llm(system_prompt: str, input_data: dict) -> dict:
    mensajes = [
        {
            "role": "system",
            "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido."
        },
        {
            "role": "user",
            "content": json.dumps(input_data)
        }
    ]

    if BACKEND_ACTUAL == "groq":
        chat_completion = cliente_groq.chat.completions.create(
            messages=mensajes,
            model=MODELO_GROQ,
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        return json.loads(chat_completion.choices[0].message.content)

    elif BACKEND_ACTUAL == "ollama":
        respuesta = ollama.chat(
            model=MODELO_OLLAMA,
            messages=mensajes,
            format="json",
            options={"temperature": 1.0}
        )
        return json.loads(respuesta['message']['content'])

In [ ]:
def indexar_politicas():
    global INDEXACION_REALIZADA

    if INDEXACION_REALIZADA:
        return

    headers_to_split_on = [
        ("#", "Titulo_Principal"),
        ("##", "Seccion"),
        ("###", "Subseccion"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    splits_estructurales = markdown_splitter.split_text(politicas_ecommerce)

    tamanos = [10, 100, 1000]
    solapamientos_pct = [0.0, 0.1, 0.8] # 0%, 10%, 80%

    textos_batch = []
    metadatos_batch = []
    ids_batch = []

    for size in tamanos:
        for pct in solapamientos_pct:
            overlap_abs = int(size * pct)
            overlap_label = int(pct * 100)

            text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="cl100k_base",
                chunk_size=size,
                chunk_overlap=overlap_abs
            )

            fragmentos = text_splitter.split_documents(splits_estructurales)

            for i, doc in enumerate(fragmentos):
                doc_id = f"chunk_s{size}_o{overlap_label}_{i}"

                meta = doc.metadata.copy()
                meta["chunk_size"] = size
                meta["overlap_pct"] = overlap_label

                textos_batch.append(doc.page_content)
                metadatos_batch.append(meta)
                ids_batch.append(doc_id)

    coleccion_politicas.upsert(
        documents=textos_batch,
        metadatas=metadatos_batch,
        ids=ids_batch
    )
    INDEXACION_REALIZADA = True
    print(f"Indexación exitosa. Documentos totales en índice: {coleccion_politicas.count()}")

In [ ]:
def recuperar_contexto(mensaje_usuario: str, k: int = 3) -> str:
    resultados = coleccion_politicas.query(
        query_texts=[mensaje_usuario],
        n_results=k,
        where={
            "$and": [
                {"chunk_size": {"$eq": RAG_CHUNK_SIZE}},
                {"overlap_pct": {"$eq": RAG_OVERLAP_PCT}}
            ]
        }
    )

    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]

    if not documentos:
        return "No se encontraron políticas relevantes."

    fragmentos_formateados = []

    for doc, meta in zip(documentos, metadatos):
        seccion = meta.get('Seccion', '')
        subseccion = meta.get('Subseccion', '')

        partes = [p for p in [seccion, subseccion] if p]

        if partes:
            encabezado = f"[{' -> '.join(partes)}]"
            fragmentos_formateados.append(f"{encabezado}\n{doc}")
        else:
            fragmentos_formateados.append(doc)

    return "\n\n--- FRAGMENTO DE POLÍTICA ---\n\n".join(fragmentos_formateados)

In [ ]:
def procesar_mensaje_con_rag(mensaje_usuario: str) -> dict:
    indexar_politicas()
    contexto_politicas = recuperar_contexto(mensaje_usuario, k=3)

    print(f"Contexto RAG recuperado. Longitud: {len(contexto_politicas)} caracteres.")

    def manejar_pausa():
        if BACKEND_ACTUAL == "groq":
            time.sleep(15)

    print(f"[{BACKEND_ACTUAL.upper()}] Procesando inferencias con LLM...")

    out_urgencia = llamar_llm(prompt_urgencia, {
        "mensaje_usuario": mensaje_usuario,
        "contexto_politicas": contexto_politicas
    })
    nivel_urgencia = out_urgencia.get("nivel_urgencia", "")
    manejar_pausa()

    out_categoria = llamar_llm(prompt_categoria, {
        "mensaje_usuario": mensaje_usuario,
        "contexto_politicas": contexto_politicas
    })
    categoria = out_categoria.get("categoria", "")
    manejar_pausa()

    out_accion = llamar_llm(prompt_accion, {
        "mensaje_usuario": mensaje_usuario,
        "nivel_urgencia": nivel_urgencia,
        "categoria": categoria,
        "contexto_politicas": contexto_politicas
    })
    cadena_de_pensamiento = out_accion.get("cadena_de_pensamiento", "")
    politica_aplicada = out_accion.get("politica_aplicada", "")
    codigo_accion_sistema = out_accion.get("codigo_accion_sistema", "")
    manejar_pausa()

    out_texto = llamar_llm(prompt_texto, {
        "mensaje_usuario": mensaje_usuario,
        "nivel_urgencia": nivel_urgencia,
        "categoria": categoria,
        "politica_aplicada": politica_aplicada
    })
    mensaje_al_cliente = out_texto.get("mensaje_al_cliente", "")
    manejar_pausa()

    out_maestro = llamar_llm(prompt_maestro, {
        "categoria": categoria,
        "nivel_urgencia": nivel_urgencia,
        "cadena_de_pensamiento": cadena_de_pensamiento,
        "politica_aplicada": politica_aplicada,
        "codigo_accion_sistema": codigo_accion_sistema,
        "mensaje_al_cliente": mensaje_al_cliente
    })

    salida_validada = SalidaMaestra(**out_maestro)
    return salida_validada.model_dump()